<a href="https://colab.research.google.com/github/surajjeoor/masai_assignments/blob/main/CNN_and_Transfer_learning.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
%pip install -q tensorflow matplotlib pillow

import os
import numpy as np
import tensorflow as tf
import matplotlib.pyplot as plt
from PIL import Image

tf.keras.utils.set_random_seed(42)

# Colab-native data loading: stable URL, no local file paths assumed
dataset_url = "https://storage.googleapis.com/download.tensorflow.org/example_images/flower_photos.tgz"
extracted_path = tf.keras.utils.get_file("flower_photos.tgz", origin=dataset_url, extract=True)
data_dir = os.path.join(extracted_path, "flower_photos")
print(data_dir)
class_names = sorted(d for d in os.listdir(data_dir) if os.path.isdir(os.path.join(data_dir, d)))
counts = {c: len(os.listdir(os.path.join(data_dir, c))) for c in class_names}

print("Classes:", class_names)
print("Image counts per class:", counts)
print("Total images:", sum(counts.values()))


228813984/228813984 ━━━━━━━━━━━━━━━━━━━━ 2s 0us/step
/root/.keras/datasets/flower_photos_extracted/flower_photos
Classes: ['daisy', 'dandelion', 'roses', 'sunflowers', 'tulips']
Image counts per class: {'daisy': 633, 'dandelion': 898, 'roses': 641, 'sunflowers': 699, 'tulips': 799}
Total images: 3670


In [3]:
img_size=(64,64)
batch_size=32
project_classes=["roses","sunflowers"]

train_ds=tf.keras.utils.image_dataset_from_directory(data_dir,validation_split=0.2,
                                                     subset="training",seed=42,image_size=img_size,
                                                     batch_size=batch_size,class_names=project_classes)

val_ds=tf.keras.utils.image_dataset_from_directory(data_dir,validation_split=0.2,
                                                   subset="validation",seed=42,image_size=img_size,
                                                   batch_size=batch_size,class_names=project_classes)
found_classes=train_ds.class_names
train_batches=tf.data.experimental.cardinality(train_ds)
val_batches=tf.data.experimental.cardinality(val_ds)

train_ds=train_ds.prefetch(tf.data.AUTOTUNE)
val_ds=val_ds.prefetch(tf.data.AUTOTUNE)

print("Found classes:",found_classes)
print("Train batches:",train_batches)
print("Val batches:",val_batches)

Found 1340 files belonging to 2 classes.
Using 1072 files for training.
Found 1340 files belonging to 2 classes.
Using 268 files for validation.
Found classes: ['roses', 'sunflowers']
Train batches: tf.Tensor(34, shape=(), dtype=int64)
Val batches: tf.Tensor(9, shape=(), dtype=int64)


In [5]:
model=tf.keras.Sequential([
    tf.keras.layers.Input((64,64,3)),
    tf.keras.layers.Rescaling(1./255),
    tf.keras.layers.Conv2D(16,3,activation="relu"),
    tf.keras.layers.MaxPooling2D(),
    tf.keras.layers.Conv2D(32,3,activation="relu"),
    tf.keras.layers.MaxPooling2D(),
    tf.keras.layers.Flatten(),
    tf.keras.layers.Dense(32,activation="relu"),
    tf.keras.layers.Dense(1,activation="sigmoid")
])

#initialize optimization parameters
model.compile(optimizer=tf.keras.optimizers.Adam(),
              loss=tf.keras.losses.BinaryCrossentropy(),
              metrics=["accuracy"])

#train
history=model.fit(train_ds,validation_data=val_ds,epochs=5,verbose=2)

#evaluation
val_loss,val_acc=model.evaluate(val_ds,verbose=0)
print(f"\n Validation accuracy:{val_acc:.4f}, Validation loss:{val_loss:.4f}")

Epoch 1/5
34/34 - 5s - 138ms/step - accuracy: 0.7593 - loss: 0.4791 - val_accuracy: 0.8881 - val_loss: 0.2539
Epoch 2/5
34/34 - 5s - 137ms/step - accuracy: 0.8834 - loss: 0.2751 - val_accuracy: 0.9179 - val_loss: 0.2215
Epoch 3/5
34/34 - 3s - 99ms/step - accuracy: 0.8909 - loss: 0.2430 - val_accuracy: 0.9179 - val_loss: 0.2161
Epoch 4/5
34/34 - 5s - 159ms/step - accuracy: 0.9058 - loss: 0.2291 - val_accuracy: 0.9142 - val_loss: 0.1989
Epoch 5/5
34/34 - 4s - 131ms/step - accuracy: 0.9254 - loss: 0.1950 - val_accuracy: 0.9104 - val_loss: 0.2106

 Validation accuracy:0.9104, Validation loss:0.2106
